# 01 • Tenseurs, gradients et taux d’apprentissage

`[MÉTA | Formation 4-024 | Niveau Application | TP 01 | Mode CPU local]`

**Objectif :** Relier les formes numériques, la dérivée et l’optimisation.

**Temps indicatif :** 30 min, plus reprise possible. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP 00.

**Preuves de réussite :** Formes justifiées, erreur attendue interceptée, trois taux comparés.

**Sources :** R02 ; dossier source pour le jouet mathématique.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [1]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


Moteur disponible : 2.10.0+cpu | Données : /mnt/data/deep_learning_4_024/03_Travaux_pratiques/donnees


## 1. Dimensions
Une matrice (2,3) multipliée par (3,4) produit (2,4). La dimension intérieure est contractée. Prédire le résultat avant l’exécution.

In [2]:
a=torch.arange(6,dtype=torch.float32).reshape(2,3)
b=torch.ones(3,4)
resultat=a@b
assert tuple(resultat.shape)==(2,4)
print(a, "\nRésultat :",resultat)
try:
    a@torch.ones(2,4)
except RuntimeError as erreur:
    print("Erreur volontaire correctement détectée :",str(erreur).split("\n")[0])

tensor([[0., 1., 2.],
        [3., 4., 5.]]) 
Résultat : tensor([[ 3.,  3.,  3.,  3.],
        [12., 12., 12., 12.]])
Erreur volontaire correctement détectée : mat1 and mat2 shapes cannot be multiplied (2x3 and 2x4)


## 2. Un neurone calculé à la main
Entrées 2 et 1 ; poids 0,5 et −1 ; biais 0,2. Calculer la somme puis appliquer ReLU. **Attendu :** 0,2 à la précision numérique près.

In [3]:
x=torch.tensor([2.0,1.0]);poids=torch.tensor([.5,-1.0]);biais=.2
z=x@poids+biais;sortie=torch.relu(z)
assert abs(sortie.item()-.2)<1e-6
print("Somme",z.item(),"Activation",sortie.item())

Somme 0.20000000298023224 Activation 0.20000000298023224


## 3. Optimiser L(w) = (w − 3)²
À chaque pas : effacer le gradient, recalculer la perte, dériver puis mettre à jour hors du graphe. Comparer un taux très faible, un taux convergent et un taux divergent. Pour ce jouet précis, 0 < taux < 1 converge ; ce n’est pas une règle générale des réseaux.

In [4]:
def trajectoire(taux, pas=20):
    w=torch.tensor(-4.0,requires_grad=True)
    lignes=[]
    for i in range(pas):
        if w.grad is not None:w.grad.zero_()
        perte=(w-3)**2
        perte.backward()
        lignes.append((i,w.item(),perte.item(),w.grad.item()))
        with torch.no_grad():w-=taux*w.grad
    return pd.DataFrame(lignes,columns=["étape","poids","perte","gradient"])
traces={t:trajectoire(t) for t in [.01,.2,1.05]}
fig,ax=plt.subplots(figsize=(7,4))
for t,d in traces.items():ax.plot(d["étape"],d["perte"],label=f"Taux {t}")
ax.set(yscale="log",xlabel="Étape",ylabel="Perte (échelle logarithmique)",title="Même objectif, trois taux")
ax.legend();fig.tight_layout();fig.savefig(RESULTS/"01_taux.png",dpi=150)
assert traces[.2].iloc[-1].perte < traces[.2].iloc[0].perte
print(traces[.2].tail())

    étape     poids         perte  gradient
15     15  2.996709  1.083311e-05 -0.006583
16     16  2.998025  3.899918e-06 -0.003950
17     17  2.998815  1.404084e-06 -0.002370
18     18  2.999289  5.054701e-07 -0.001422
19     19  2.999573  1.819286e-07 -0.000853


## 4. Dérivée composée
Calculer y = (2w + 1)² en w = 1. La règle de chaîne donne 2 × (2w + 1) × 2 = 12.

**Interprétation demandée :** quelle opération apporte chaque facteur ?

In [5]:
w=torch.tensor(1.,requires_grad=True)
y=(2*w+1)**2;y.backward()
assert w.grad.item()==12.
print("Gradient composé :",w.grad.item())

Gradient composé : 12.0


## 5. Preuve et remédiation
Expliquer les axes ; distinguer poids appris et taux choisi ; justifier une convergence lente sans accuser automatiquement la capacité du modèle.

**Extension :** comparer le gradient automatique à une différence finie pour une perturbation de 0,001. Une différence finie trop petite en précision simple peut devenir numériquement fragile.